In [0]:
from pyspark.sql.functions import col, lit, trim, coalesce

In [0]:
bronze = "abfss://bronze@databricktraveljournal.dfs.core.windows.net"
table = "google_maps_address"
bronze_table_parquet_path = f"{bronze}/{table}/"

google_maps_address_df = spark.read.format("parquet")\
    .load(f"{bronze_table_parquet_path}")



In [0]:
google_maps_address_df.printSchema()

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, trim, coalesce
from pyspark.sql.functions import col, coalesce, get_json_object, to_timestamp, lit, when


def transform_to_silver(bronze_df: DataFrame) -> DataFrame:
    df = bronze_df

    # 1. Parse timestamps


    df = df.withColumn("created_at",
                               when(col("created_at").isNull(),
                                    get_json_object(col("_rescued_data"),"$.created_at"))
                                    .otherwise(col("created_at"))
                                    )
    df = (
        df
        # if the year is 2024, replace it with 2026 (keeps month/day/time exactly)
        .withColumn(
            "created_at",
            F.when(
                F.col("created_at").startswith("2024"),
                F.regexp_replace("created_at", r"^2024", "2026")
            ).otherwise(F.col("created_at"))
        )
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("date_type", F.to_date("created_at"))
    )
    df = (
        df.withColumn("created_at", F.to_timestamp("created_at"))
          .withColumn("date_type", F.to_date("date_type"))
    )

    # 2. Normalize flag string -> boolean
    df = df.withColumn("flag", F.lower(F.trim(F.col("flag"))) == F.lit("true"))

    # 3. Trim string columns
    df = (
        df.withColumn("address",        trim(col("address")))
          .withColumn("city",           trim(col("city")))
          .withColumn("country",        trim(col("country")))
          .withColumn("district",       trim(col("district")))
          .withColumn("name",           trim(col("name")))
          .withColumn("neighborhood",   trim(col("neighborhood")))
          .withColumn("postal_code",    trim(col("postal_code")))
          .withColumn("state_province", trim(col("state_province")))
          .withColumn("street",         trim(col("street")))
          .withColumn("year",  F.when(F.col("year") == 2024, F.lit(2026)).otherwise(F.col("year")))

    )

    # 4. Cast types (do this BEFORE validating ranges)
    df = (
        df.withColumn("latitude",  F.expr("try_cast(latitude as double)"))
          .withColumn("longitude", F.expr("try_cast(longitude as double)"))
          .withColumn("year",      F.expr("try_cast(year as int)"))
          .withColumn("month",     F.expr("try_cast(month as int)"))
          .withColumn("day",       F.expr("try_cast(day as int)"))
    )

    # 5. Quality flag (coalesce nulls to False)
    quality_df = df.withColumn(
        "_is_valid",
        coalesce(
            col("id").isNotNull()
            & col("place_id").isNotNull()
            & col("created_at").isNotNull()
            & col("latitude").isNotNull()
            & col("longitude").isNotNull()
            & col("city").isNotNull()
            & col("country").isNotNull()
            & col("name").isNotNull()
            & col("latitude").between(-90, 90)
            & col("longitude").between(-180, 180),
            lit(False),
        ),
    )

    # 6. Split valid / invalid
    valid_df   = quality_df.filter(col("_is_valid"))
    invalid_df = quality_df.filter(~col("_is_valid"))

    # 7. Log quarantined rows once
    invalid_count = invalid_df.count()
    if invalid_count > 0:
        print(f"Quarantined {invalid_count} invalid records")

    return valid_df.drop("_rescued_data","_is_valid")


df = transform_to_silver(google_maps_address_df)

## Deduplicate


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def deduplicate_by_key(df, keycolumns, order_column, ascending=False):
    """
    
    Deduplicate a Dataframe by composite key, keeping the row with the highest (or lowest) value in order_column

    Args:
        df: Input DataFrame with duplicates
        key_columns: List of columns forming the composite key
        order_column: Column to break ties (e.g., updated_at)
        ascending: If True, keep the smallest order_column value
    """
    try:

        order_expr = (
            F.col(order_column).asc() if ascending else F.col(order_column).desc()
        )

        window_spec = Window.partitionBy(*keycolumns).orderBy(order_expr)

        return df.withColumn("rank", F.row_number().over(window_spec)).filter(
            F.col("rank") == 1
        ).drop("rank")
    except ValueError as e:
        print("Error:", e)


google_maps_address_df = deduplicate_by_key(df, ["id"], "created_at", ascending=True)



### Data Writing


In [0]:
google_maps_address_df.write.format("delta").mode("overwrite").save("abfss://silver@databricktraveljournal.dfs.core.windows.net/google_maps_address")

## DELTA

In [0]:
%sql

CREATE TABLE IF NOT EXISTS travel_journal_catalog.silver.google_maps_address_silver 
USING DELTA
LOCATION "abfss://silver@databricktraveljournal.dfs.core.windows.net/google_maps_address"